# Rotations, Orientations, And Batch Primitives

PyTex distinguishes between a mathematical rotation and an orientation that lives
between a crystal frame and a specimen frame. This notebook also shows how the batch
primitives let large vectorized workflows stay semantically typed.

## Key Mathematical Ideas

A unit quaternion is stored in `w, x, y, z` order and defines the active rotation

$$
\mathbf{v}' = \mathbf{R}(q)\,\mathbf{v}.
$$

An `Orientation` then wraps that rotation together with the frame pair
`(crystal_frame, specimen_frame)`.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, *_ = make_context()

eulers = np.array(
    [
        [0.0, 0.0, 0.0],
        [45.0, 35.0, 15.0],
        [90.0, 0.0, 0.0],
    ]
)

euler_set = EulerSet(eulers, convention="bunge", degrees=True)
rotation_set = euler_set.to_rotation_set()
quaternion_set = rotation_set.as_quaternion_set()
orientation_set = OrientationSet.from_euler_angles(
    euler_set,
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=crystal),
)

print("Euler angles")
print(euler_set.as_array())
print("Quaternions")
print(quaternion_set.as_array())


In [ ]:
crystal_vectors = VectorSet(
    values=np.array(
        [
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
        ]
    ),
    reference_frame=crystal,
)

specimen_vectors = orientation_set.map_crystal_directions(crystal_vectors)
print(specimen_vectors.reference_frame.name)
print(specimen_vectors.values)


## Batch Semantics

The important point is not just that these operations are vectorized. It is that the
shared convention or frame survives the operation:

- `EulerSet` retains the Euler convention
- `QuaternionSet` retains canonical quaternion normalization
- `RotationSet` rotates many vectors at once
- `OrientationSet` carries crystal/specimen meaning while doing batched direction maps
